# Entrenar la IA de DataGuard (GLiNER)

Ajusta (*fine-tuning*) el modelo `urchade/gliner_multi_pii-v1` con ejemplos **ficticios** parecidos a lo que ve DataGuard
(mails, chats, formularios y prompts de empresas argentinas) y lo deja listo para la app web: ONNX comprimido de ~300 MB,
mismo tamaño y velocidad que el actual.

**Antes de empezar**

1. En el proyecto, generá los ejemplos: `npm run entrenamiento:datos` → crea `.cache/entrenamiento/train.json` y `dev.json`.
2. Acá arriba: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** (gratis).
3. **Entorno de ejecución → Ejecutar todas**. Te va a pedir los dos archivos y, al final, permiso para guardar en tu Google Drive.

Con GPU tarda unos 30-45 minutos. Solo se suben datos inventados: **nunca pongas datos reales** de la empresa en los ejemplos.

## 1. Entorno

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gliner==0.2.29", "onnx", "onnxruntime"], check=True)

import json, math, shutil, urllib.request
import onnx
import torch
from gliner import GLiNER

IN_COLAB = "google.colab" in sys.modules
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} · dispositivo: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
if DEVICE == "cpu":
    print("⚠ Sin GPU el entrenamiento tarda horas: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4")

## 2. Configuración

Los valores por defecto son un buen punto de partida. Si después de entrenar **empeora** en `npm run eval:ia`,
lo primero es bajar `EPOCAS` a 1 o `LR_ENCODER` a `3e-6`: el modelo se está acomodando demasiado a los ejemplos.

In [ ]:
MODELO_BASE = "urchade/gliner_multi_pii-v1"  # el original del que sale el ONNX que usa la web
ETIQUETAS = ["person", "organization", "address", "medical condition"]  # = LABEL_NAMES en packages/ml/src/model.ts
EPOCAS = 2  # pasadas completas por train.json
LOTE = 8  # ejemplos por paso
LR_ENCODER = 5e-6  # bajo a propósito: aprende lo nuevo sin olvidar lo que ya sabía
LR_RESTO = 1e-5  # capas propias de GLiNER (spans y etiquetas)
UMBRAL = 0.5  # solo para medir antes/después acá (la web usa umbrales calibrados por etiqueta)
GUARDAR_COMPLETO = False  # True: también guarda el modelo sin comprimir (1,2 GB), por si querés probar otras compresiones
REPO_RAW = "https://raw.githubusercontent.com/Ju4nse/dataguard/main/"

# Prueba rápida: pocos ejemplos y 20 pasos, solo para verificar que todo el circuito anda (no sirve como modelo).
PRUEBA_RAPIDA = os.environ.get("PRUEBA_RAPIDA") == "1"

## 3. Datos

In [ ]:
faltan = [f for f in ("train.json", "dev.json") if not os.path.exists(f)]
if faltan and IN_COLAB:
    from google.colab import files

    print("Elegí train.json y dev.json (en el proyecto están en .cache/entrenamiento/)")
    files.upload()
    faltan = [f for f in ("train.json", "dev.json") if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan {faltan}: generalos con npm run entrenamiento:datos y subilos con el mismo nombre")

with open("train.json", encoding="utf-8") as f:
    train = json.load(f)
with open("dev.json", encoding="utf-8") as f:
    dev = json.load(f)
if PRUEBA_RAPIDA:
    train, dev = train[:64], dev[:32]

for nombre, datos in (("train", train), ("dev", dev)):
    conteo = {e: 0 for e in ETIQUETAS}
    for ej in datos:
        for _, _, etiqueta in ej["ner"]:
            conteo[etiqueta] += 1
    print(f"{nombre}: {len(datos)} ejemplos · {conteo}")
ej = train[0]
print("Ejemplo:", " ".join(ej["tokenized_text"]))
print("  →", [(" ".join(ej["tokenized_text"][a : b + 1]), etiqueta) for a, b, etiqueta in ej["ner"]])

## 4. Modelo original y medición de partida

`dev.json` usa plantillas que el entrenamiento nunca ve: mide si el modelo generaliza o solo se aprendió las frases.
Las frases de abajo son de control a ojo; las dos últimas **no** deberían marcar nada.

In [ ]:
model = GLiNER.from_pretrained(MODELO_BASE).to(DEVICE)

FRASES = [
    "Hola, soy Brenda Kowalczyk de Aceitera Los Talas SRL; mandame la cotización.",
    "El chofer Ricardo Insúa tuvo una luxación de hombro y está con licencia.",
    "Entregar en Bv. Rondeau 2240, Marcos Paz, a nombre de la Sra. Quintana.",
    "Gerencia de Compras pidió que Logística revise el pedido en Excel.",
    "La reunión con AFIP es el martes en Córdoba.",
]


def medir(titulo):
    salida, f1 = model.evaluate(dev, flat_ner=True, threshold=UMBRAL, batch_size=16, entity_types=ETIQUETAS)
    print(f"{titulo}: {salida}")
    return f1


def mostrar(titulo, m=None):
    print(titulo)
    resultados = []
    for frase in FRASES:
        ents = (m or model).predict_entities(frase, ETIQUETAS, threshold=UMBRAL, flat_ner=True)
        resultados.append([(e["text"], e["label"]) for e in ents])
        print(" ·", frase, "→", [(e["text"], e["label"], round(e["score"], 2)) for e in ents] or "nada")
    return resultados


f1_antes = medir("Antes de entrenar (dev)")
mostrar("Antes:");

## 5. Entrenar

In [ ]:
pasos = 20 if PRUEBA_RAPIDA else math.ceil(len(train) / LOTE) * EPOCAS
args = model.create_training_args(
    output_dir="checkpoints",
    learning_rate=LR_ENCODER,
    others_lr=LR_RESTO,
    per_device_train_batch_size=LOTE,
    per_device_eval_batch_size=LOTE,
    max_steps=pasos,
    warmup_ratio=0.1,
    logging_steps=5 if PRUEBA_RAPIDA else 50,
    save_strategy="no",  # sin checkpoints intermedios (ocupan GB): el modelo entrenado queda en memoria
    bf16=DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8,  # solo en GPUs que lo soportan (no la T4)
    use_cpu=DEVICE == "cpu",
    dataloader_num_workers=0,
)
print(f"Entrenando {pasos} pasos ({len(train)} ejemplos, lote {LOTE})…")
model.train_model(train_dataset=train, eval_dataset=dev, training_args=args);

## 6. Comparar

In [ ]:
f1_despues = medir("Después de entrenar (dev)")
frases_torch = mostrar("Después:")
print(f"\nF1 en dev: {f1_antes:.3f} → {f1_despues:.3f}")
if f1_despues < f1_antes:
    print("⚠ Empeoró en dev: no lo uses. Probá con EPOCAS = 1 o LR_ENCODER = 3e-6.")

## 7. Exportar a ONNX y comprimir

Mismo formato que el modelo actual de la web (mismas entradas) y la misma compresión "solo pesos" de
`scripts/quantize-model.py --plegado`. Después se carga el archivo comprimido y se compara con el modelo en memoria.

In [ ]:
model = model.to("cpu").eval()
model.export_to_onnx("export", onnx_filename="model.onnx", quantize=False, opset=14)

if not os.path.exists("quantize-model.py"):
    urllib.request.urlretrieve(REPO_RAW + "scripts/quantize-model.py", "quantize-model.py")
subprocess.run(
    [sys.executable, "quantize-model.py", "--plegado", "--entrada", "export/model.onnx", "--salida", "export/model_ft_w8p.onnx"],
    check=True,
)

entradas = [i.name for i in onnx.load("export/model_ft_w8p.onnx", load_external_data=False).graph.input]
assert entradas == ["input_ids", "attention_mask", "words_mask", "text_lengths", "span_idx", "span_mask"], entradas

comprimido = GLiNER.from_pretrained("export", load_onnx_model=True, onnx_model_file="model_ft_w8p.onnx", local_files_only=True)
frases_onnx = mostrar("Modelo comprimido (ONNX):", comprimido)
iguales = sum(a == b for a, b in zip(frases_torch, frases_onnx))
print(f"\nMismo resultado que sin comprimir en {iguales} de {len(FRASES)} frases")

## 8. Guardar en Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    destino = "/content/drive/MyDrive/dataguard-modelo"
else:
    destino = "modelo-entrenado"
os.makedirs(destino, exist_ok=True)
archivos = ["model_ft_w8p.onnx", "gliner_config.json"] + (["model.onnx"] if GUARDAR_COMPLETO else [])
for archivo in archivos:
    shutil.copy(os.path.join("export", archivo), destino)
    print(f"✓ {destino}/{archivo} ({os.path.getsize(os.path.join(destino, archivo)) / 1e6:.0f} MB)")

## 9. Probarlo en DataGuard (en tu PC)

1. Bajá `model_ft_w8p.onnx` de tu Drive (carpeta `dataguard-modelo`) a `.cache/modelos/gliner_multi_pii-v1/onnx/` del proyecto.
2. Compará contra el modelo actual:
   ```
   npm run eval:ia
   npm run eval:ia -- --modelo onnx/model_ft_w8p.onnx --errores
   ```
3. Si mejora, recalibrá los umbrales: `npm run calibrar:ia -- --modelo onnx/model_ft_w8p.onnx` (y actualizá `LABEL_THRESHOLDS`).
4. Examen final, **una sola vez**: `npm run eval:ia -- --modelo onnx/model_ft_w8p.onnx --control`.
   Si no le gana al modelo actual en el set de control, no se usa.
5. Si gana: hay que publicarlo (Hugging Face o un release de GitHub) y apuntar `MODEL` en `packages/ml/src/model.ts`.